# 질문 유형별 전문 AI 앙상블 (Mixture-of-Experts)

질문 유형을 먼저 분류한 뒤, **유형별 전용 LoRA 어댑터**로 라우팅하는 방식입니다.

## 전략 요약
```
질문 입력 → 유형 분류기 → 전문 모델 라우팅 → 정답 출력
          (rule-based)    (객체/수량/속성/...)   
```

| 유형 | 키워드 | 특화 포인트 |
|------|--------|-------------|
| 무엇 | 무엇, 뭐 | 객체 인식, 재질 분류 |
| 몇 개 | 몇, 개수 | 카운팅, 수량 파악 |
| 어떤 | 어떤, 종류 | 속성·분류 구분 |
| 어디 | 어디, 위치 | 공간·위치 파악 |
| 색깔 | 색, 색깔 | 색상 인식 |
| 어떻게 | 어떻게, 방법 | 방법·과정 추론 |
| 누구 | 누구, 사람 | 인물 식별 |
| 참거짓 | 옳은, 맞는 | True/False 판단 |

# 1. 환경 준비

In [ ]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

In [ ]:
# Qwen3-VL은 아직 pip 정식 버전 미지원 → git 최신 버전 필요
!pip install -q git+https://github.com/huggingface/transformers
!pip -q install "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade

# 2. 라이브러리, 데이터, 설정

In [ ]:
import os, re, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from collections import Counter
from transformers import (
    Qwen3VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# ── 기본 설정 ──────────────────────────────────────────────────
MODEL_ID      = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE    = 384
MAX_NEW_TOKENS = 8
SEED          = 42
EXP_NUM       = 1   # ← 새 실험마다 여기만 변경!
BASE_SAVE_DIR = f"./qwen3_exp{EXP_NUM}_experts"  # 유형별 어댑터 저장 루트

random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

print(f"학습 데이터: {len(train_df)}개")
print(f"테스트 데이터: {len(test_df)}개")

# 3. 질문 유형 분류기

Rule-based 분류기로 질문 텍스트에서 유형을 판별합니다.
우선순위 순서로 매칭하며, 마지막은 `기타`로 처리합니다.

In [ ]:
# 질문 유형 정의 및 우선순위 키워드
QUESTION_TYPES = [
    ("색깔",    ["색깔", "색상", "색"]),
    ("몇개",    ["몇 개", "몇개", "몇 가지", "몇가지", "몇"]),
    ("어디",    ["어디", "어느 위치", "위치"]),
    ("어떻게",  ["어떻게", "방법", "방식"]),
    ("누구",    ["누구", "누가", "사람"]),
    ("참거짓",  ["옳은", "맞는", "올바른", "잘못된", "틀린"]),
    ("어떤",    ["어떤", "어느"]),
    ("무엇",    ["무엇", "뭐", "뭔가", "무슨"]),
]
TYPE_OTHER = "기타"
ALL_TYPES = [t for t, _ in QUESTION_TYPES] + [TYPE_OTHER]


def classify_question(question: str) -> str:
    """질문 텍스트 → 유형 문자열 반환"""
    q = question.lower()
    for type_name, keywords in QUESTION_TYPES:
        for kw in keywords:
            if kw in q:
                return type_name
    return TYPE_OTHER


# ── 데이터에 유형 컬럼 추가 ────────────────────────────────────
train_df["q_type"] = train_df["question"].apply(classify_question)
test_df["q_type"]  = test_df["question"].apply(classify_question)

print("📊 Train 유형 분포:")
print(train_df["q_type"].value_counts())

# 실제 존재하는 유형만 학습 대상으로 선정
ACTIVE_TYPES = train_df["q_type"].unique().tolist()
print(f"\n✅ 활성 유형 {len(ACTIVE_TYPES)}개: {ACTIVE_TYPES}")

# 4. 모델 & Processor 초기화

베이스 모델은 **한 번만** 로드합니다.  
각 유형 전문가는 **별도 LoRA 어댑터**만 교체합니다.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    trust_remote_code=True,
)
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 설정 (모든 전문가 공통)
LORA_CONFIG = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

print("✅ 베이스 모델 로드 완료")

# 5. 유형별 시스템 프롬프트 정의

각 전문가는 해당 유형에 특화된 시스템 지시사항을 사용합니다.

In [ ]:
# 유형별 전용 시스템 프롬프트
SYSTEM_PROMPTS = {
    "무엇": (
        "You are an expert at object recognition and material identification. "
        "Focus on identifying what the object IS, its type, and material. "
        "당신은 사물 인식 전문가입니다. 이미지에서 사물의 종류와 재질을 정확히 파악하세요. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "몇개": (
        "You are an expert at counting objects in images. "
        "Carefully count every instance of the specified object. "
        "당신은 이미지에서 물체를 세는 전문가입니다. 지정된 물체의 개수를 정확히 세세요. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "어떤": (
        "You are an expert at classifying attributes and properties of objects. "
        "Focus on the specific property, type, or category being asked about. "
        "당신은 사물의 속성과 종류를 분류하는 전문가입니다. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "어디": (
        "You are an expert at spatial reasoning and location identification. "
        "Focus on WHERE objects are positioned relative to each other or to the scene. "
        "당신은 공간 추론과 위치 파악 전문가입니다. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "색깔": (
        "You are an expert at color recognition and identification. "
        "Carefully examine the color of the specified object or area. "
        "당신은 색상 인식 전문가입니다. 이미지에서 특정 물체나 영역의 색상을 정확히 파악하세요. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "어떻게": (
        "You are an expert at process and method reasoning from visual context. "
        "Focus on HOW something is done, categorized, or organized in the image. "
        "당신은 방법과 과정을 시각적으로 추론하는 전문가입니다. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "누구": (
        "You are an expert at identifying persons and human attributes in images. "
        "Focus on the people, their roles, or activities shown. "
        "당신은 이미지에서 인물을 식별하는 전문가입니다. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "참거짓": (
        "You are an expert at visual fact-checking and true/false reasoning. "
        "Carefully verify which statement is TRUE based on what you see. "
        "당신은 시각적 사실 검증 전문가입니다. 이미지를 바탕으로 옳은 설명을 선택하세요. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
    "기타": (
        "You are an expert visual question answering assistant. "
        "Carefully examine the image and all answer choices before responding. "
        "당신은 이미지를 보고 질문에 답하는 전문가입니다. "
        "정답을 a, b, c, d 중 하나의 소문자로만 답하세요."
    ),
}


def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요."
    )


print("✅ 유형별 시스템 프롬프트 설정 완료")
for qtype, prompt in SYSTEM_PROMPTS.items():
    print(f"  [{qtype}] {prompt[:60]}...")

# 6. Dataset & Collator

In [ ]:
class VQAMCDataset(Dataset):
    """특정 유형의 데이터만 담는 Dataset (system_prompt 자동 선택)"""

    def __init__(self, df, processor, q_type, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.q_type = q_type
        self.train = train
        self.system_prompt = SYSTEM_PROMPTS.get(q_type, SYSTEM_PROMPTS["기타"])

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")
        user_text = build_mc_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )
        messages = [
            {"role": "system", "content": [{"type": "text", "text": self.system_prompt}]},
            {"role": "user",   "content": [
                {"type": "image", "image": img},
                {"type": "text",  "text": user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role": "assistant", "content": [{"type": "text", "text": gold}]})

        return {"messages": messages, "image": img}


@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"], tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts, images=images, padding=True, return_tensors="pt"
        )
        if self.train:
            enc["labels"] = enc["input_ids"].clone()
        return enc


print("✅ Dataset / Collator 정의 완료")

# 7. 정답 추출 함수

In [ ]:
def extract_choice(text: str) -> str:
    text = text.strip().lower()
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if not lines:
        return "a"
    last = lines[-1]
    if last in ["a", "b", "c", "d"]:
        return last
    for line in reversed(lines):
        for tok in line.split():
            tok_clean = tok.strip(".,!?()[]")
            if tok_clean in ["a", "b", "c", "d"]:
                return tok_clean
    m = re.search(r'\b([abcd])\b', text)
    return m.group(1) if m else "a"


print("✅ extract_choice 정의 완료")

# 8. 유형별 전문 모델 학습

각 질문 유형에 대해 **개별 LoRA 어댑터**를 학습합니다.

```
for 유형 in ACTIVE_TYPES:
    1. 해당 유형 데이터만 분리
    2. 새 LoRA 어댑터 장착
    3. 학습 실행
    4. 어댑터 저장 (BASE_SAVE_DIR/{유형}/)
```

In [ ]:
# ── 학습 하이퍼파라미터 ────────────────────────────────────────
EPOCHS     = 1        # 유형별 에폭 수 (데이터가 적으므로 1~2 권장)
LR         = 2e-4
GRAD_ACCUM = 4
AMP_DTYPE  = torch.bfloat16

# 유형별 최소 샘플 수 기준 (너무 적으면 기타 모델로 대체)
MIN_SAMPLES = 50


def train_expert(q_type: str, type_df: pd.DataFrame) -> str:
    """
    특정 유형 전문 어댑터 학습.
    Returns: 저장 경로 (str) or None (샘플 부족)
    """
    save_dir = os.path.join(BASE_SAVE_DIR, q_type)

    if len(type_df) < MIN_SAMPLES:
        print(f"  ⚠ [{q_type}] 샘플 {len(type_df)}개 < {MIN_SAMPLES} → 학습 건너뜀 (기타 모델 사용)")
        return None

    print(f"\n{'='*60}")
    print(f"  🔹 [{q_type}] 전문가 학습 시작  (샘플 {len(type_df)}개)")
    print(f"{'='*60}")

    # Train / Valid 분리
    shuffled = type_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    split = int(len(shuffled) * 0.9)
    train_sub = shuffled.iloc[:split].reset_index(drop=True)
    valid_sub  = shuffled.iloc[split:].reset_index(drop=True)
    print(f"  Train: {len(train_sub)}개 / Valid: {len(valid_sub)}개")

    # DataLoader
    train_ds     = VQAMCDataset(train_sub, processor, q_type, train=True)
    valid_ds     = VQAMCDataset(valid_sub,  processor, q_type, train=True)
    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                              collate_fn=DataCollator(processor, True), num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=1, shuffle=False,
                              collate_fn=DataCollator(processor, True), num_workers=0)

    # 새 LoRA 어댑터 장착
    expert_model = get_peft_model(base_model, LORA_CONFIG)
    expert_model.print_trainable_parameters()
    expert_model.train()

    # Optimizer / Scheduler
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, expert_model.parameters()),
        lr=LR, weight_decay=0.01
    )
    total_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps
    )
    scaler = torch.amp.GradScaler('cuda', enabled=(AMP_DTYPE == torch.float16))

    best_val_loss = float("inf")

    for epoch in range(EPOCHS):
        expert_model.train()
        running = 0.0
        global_step = 0
        optimizer.zero_grad(set_to_none=True)

        progress_bar = tqdm(train_loader,
                            desc=f"[{q_type}] Epoch {epoch+1} train",
                            unit="batch")

        for step, batch in enumerate(progress_bar, start=1):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE):
                outputs = expert_model(**batch)
                loss = outputs.loss / GRAD_ACCUM

            if scaler.is_enabled():
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running += loss.item()

            if step % GRAD_ACCUM == 0:
                if scaler.is_enabled():
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1
                progress_bar.set_postfix({"loss": f"{running/GRAD_ACCUM:.3f}"})
                running = 0.0

        # Validation
        expert_model.eval()
        val_loss, val_steps = 0.0, 0
        with torch.no_grad(), torch.amp.autocast('cuda', dtype=AMP_DTYPE):
            for vb in tqdm(valid_loader, desc=f"[{q_type}] Epoch {epoch+1} valid", unit="batch"):
                vb = {k: v.to(device) for k, v in vb.items()}
                val_loss += expert_model(**vb).loss.item()
                val_steps += 1
        val_loss_avg = val_loss / max(val_steps, 1)
        print(f"  [Epoch {epoch+1}] [{q_type}] valid loss {val_loss_avg:.4f}")

        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg
            os.makedirs(save_dir, exist_ok=True)
            expert_model.save_pretrained(save_dir)
            processor.save_pretrained(save_dir)
            print(f"  ★ Best [{q_type}] saved → {save_dir} (loss={best_val_loss:.4f})")

    # LoRA 어댑터를 베이스 모델에서 분리 (메모리 해제)
    del expert_model
    torch.cuda.empty_cache()

    print(f"  ✅ [{q_type}] 학습 완료. Best Loss: {best_val_loss:.4f}")
    return save_dir


print("✅ train_expert 함수 정의 완료")

In [ ]:
# ── 모든 유형 순서대로 학습 ────────────────────────────────────
# 저장된 어댑터 경로를 딕셔너리로 관리
EXPERT_DIRS = {}   # {q_type: save_dir or None}

for q_type in ACTIVE_TYPES:
    type_df = train_df[train_df["q_type"] == q_type].reset_index(drop=True)
    save_dir = train_expert(q_type, type_df)
    EXPERT_DIRS[q_type] = save_dir

print("\n" + "="*60)
print("📦 전문가 어댑터 저장 현황:")
for qtype, d in EXPERT_DIRS.items():
    status = d if d else "→ 기타 모델로 대체 예정"
    print(f"  [{qtype}]: {status}")

# 9. 전문가 라우터 (Inference)

추론 시:
1. 질문 유형 분류
2. 해당 유형의 LoRA 어댑터 로드
3. 추론 후 어댑터 언로드 (VRAM 효율)

> 샘플이 부족해 학습되지 않은 유형은 **기타** 어댑터로 자동 대체합니다.

In [ ]:
# 유형별 배치로 그룹화해서 같은 어댑터는 한 번만 로드
# → 어댑터 교체 횟수를 최소화해 inference 속도 향상

# 폴백 어댑터: 기타 or 가장 큰 유형
FALLBACK_TYPE = "기타" if EXPERT_DIRS.get("기타") else max(
    EXPERT_DIRS, key=lambda k: 0 if EXPERT_DIRS[k] is None else
    len(train_df[train_df["q_type"] == k])
)
print(f"폴백 유형: {FALLBACK_TYPE}")


def get_expert_dir(q_type: str) -> str:
    """유형에 맞는 어댑터 경로 반환. 없으면 폴백."""
    d = EXPERT_DIRS.get(q_type)
    if d and os.path.isdir(d):
        return d
    fallback_d = EXPERT_DIRS.get(FALLBACK_TYPE)
    if fallback_d and os.path.isdir(fallback_d):
        return fallback_d
    raise FileNotFoundError(f"어댑터를 찾을 수 없음: {q_type}, {FALLBACK_TYPE}")


print("✅ 폴백 로직 설정 완료")

In [ ]:
# ── 유형별 배치 추론 ──────────────────────────────────────────
# test_df에 index를 유지한 채 유형별로 그룹화

model_eval = None   # 현재 로드된 expert 모델
current_type = None # 현재 어댑터 유형

preds_dict = {}  # {test_index: answer}

# 유형별로 인덱스 그룹 만들기
type_groups = test_df.groupby("q_type").groups  # {q_type: Index}

for q_type, indices in type_groups.items():
    adapter_dir = get_expert_dir(q_type)
    print(f"\n🔄 [{q_type}] 전문가 어댑터 로드: {adapter_dir}")
    print(f"   해당 유형 테스트 샘플: {len(indices)}개")

    # 이전 모델 해제
    if model_eval is not None:
        del model_eval
        torch.cuda.empty_cache()

    # 전문가 어댑터 로드
    model_eval = PeftModel.from_pretrained(base_model, adapter_dir)
    model_eval.eval()
    current_type = q_type

    for i in tqdm(indices, desc=f"[{q_type}] Inference", unit="sample"):
        row = test_df.loc[i]
        try:
            img = Image.open(row["path"]).convert("RGB")
            user_text = build_mc_prompt(
                row["question"], row["a"], row["b"], row["c"], row["d"]
            )
            system_prompt = SYSTEM_PROMPTS.get(q_type, SYSTEM_PROMPTS["기타"])

            messages = [
                {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
                {"role": "user",   "content": [
                    {"type": "image", "image": img},
                    {"type": "text",  "text": user_text}
                ]}
            ]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

            with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
                out_ids = model_eval.generate(
                    **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                    eos_token_id=processor.tokenizer.eos_token_id
                )
            output_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
            preds_dict[i] = extract_choice(output_text)

        except Exception as e:
            print(f"  [Sample {i}] 에러: {e}")
            preds_dict[i] = "a"

        finally:
            del inputs
            torch.cuda.empty_cache()

# 마지막 모델 해제
if model_eval is not None:
    del model_eval
    torch.cuda.empty_cache()

print("\n✅ 모든 유형 추론 완료")

# 10. 제출 파일 생성

In [ ]:
# 원본 test_df 순서대로 정렬
preds_ordered = [preds_dict[i] for i in test_df.index]

submission = pd.DataFrame({"id": test_df["id"], "answer": preds_ordered})
submission.to_csv("submission.csv", index=False)
print("✅ Saved submission.csv")
print(f"\n📊 정답 분포:\n{submission['answer'].value_counts()}")

# 유형별 예측 분포 확인
test_df_result = test_df.copy()
test_df_result["pred"] = preds_ordered
print("\n📊 유형별 예측 분포:")
print(test_df_result.groupby("q_type")["pred"].value_counts().unstack(fill_value=0))

# 11. (선택) 커널 재시작 후 Inference only

학습된 어댑터가 있을 경우, 학습 과정 없이 바로 추론만 실행합니다.

In [ ]:
# ── 저장된 전문가 어댑터 목록 자동 탐색 ─────────────────────
# 커널 재시작 후 이 셀부터 실행하세요

# BASE_SAVE_DIR = f"./qwen3_exp{EXP_NUM}_experts"  # 필요시 재정의

# EXPERT_DIRS_LOADED = {}
# if os.path.isdir(BASE_SAVE_DIR):
#     for name in os.listdir(BASE_SAVE_DIR):
#         full_path = os.path.join(BASE_SAVE_DIR, name)
#         if os.path.isdir(full_path):
#             EXPERT_DIRS_LOADED[name] = full_path
#             print(f"  ✅ [{name}] 어댑터 발견: {full_path}")

# # 이후 Section 9의 추론 코드를 EXPERT_DIRS_LOADED 기반으로 실행
print("위 주석을 해제하고 실행하세요.")